# Линейная регрессия — EdStatsCountry

**Задача:** предсказать год последних торговых данных (`Latest trade data`) по году последних промышленных данных (`Latest industrial data`).

**Идея:** если страна недавно обновила промышленную статистику — скорее всего она обновила и торговую. Проверим это через линейную регрессию.

---

## 1. Импорт библиотек

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

print('Библиотеки загружены успешно')

## 2. Загрузка и подготовка данных

In [ ]:
marks = pd.read_csv('EdStatsCountry.csv')

print(f'Загружено строк: {len(marks)}, колонок: {len(marks.columns)}')
marks.head()

In [ ]:
# Берём только нужные колонки и убираем строки с пустыми значениями
df = marks[['Latest industrial data', 'Latest trade data']].dropna()

print(f'Строк после удаления пустых: {len(df)}')
print()
print(df.describe())

## 3. Разбивка на обучающую и тестовую выборки

In [ ]:
# X — признак (то что подаём на вход модели)
# y — целевая переменная (то что хотим предсказать)
X = df[['Latest industrial data']]  # двойные скобки — pandas ожидает таблицу, не столбец
y = df['Latest trade data']

# 80% данных — для обучения, 20% — для проверки
# random_state=42 — фиксирует случайность, чтобы результат был одинаковым при каждом запуске
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Обучающая выборка:  {len(X_train)} стран')
print(f'Тестовая выборка:   {len(X_test)} стран')

## 4. Обучение модели

In [ ]:
model = LinearRegression()

# Здесь происходит обучение — модель подбирает коэффициенты
model.fit(X_train, y_train)

print(f'Коэффициент (наклон прямой): {model.coef_[0]:.4f}')
print(f'Свободный член (intercept):  {model.intercept_:.4f}')
print()
print('Формула: Latest trade data = '
      f'{model.coef_[0]:.4f} * Latest industrial data + {model.intercept_:.4f}')

## 5. Предсказания и оценка качества

In [ ]:
# Делаем предсказания на тестовой выборке
y_pred = model.predict(X_test)

# MAE — средняя ошибка в годах
mae = mean_absolute_error(y_test, y_pred)

# R² — насколько хорошо модель объясняет данные (от 0 до 1, чем выше тем лучше)
r2 = r2_score(y_test, y_pred)

# Baseline MAE — ошибка если всегда предсказывать среднее (без модели)
baseline_mae = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))

print('=' * 50)
print('РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ')
print('=' * 50)
print(f'MAE модели:      {mae:.2f} лет')
print(f'MAE без модели:  {baseline_mae:.2f} лет  (если всегда говорить "среднее")')
print(f'R² score:        {r2:.4f}')
print()
if mae < baseline_mae:
    print(f'✓ Модель лучше baseline на {baseline_mae - mae:.2f} лет')
else:
    print('✗ Модель не лучше чем просто среднее — данных слишком мало')

## 6. Визуализация

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── График 1: данные + линия регрессии ──────────────────────────────────────
ax1 = axes[0]
ax1.scatter(X_test, y_test, color='steelblue', alpha=0.7, label='Реальные данные')
ax1.plot(X_test, y_pred, color='red', linewidth=2, label='Линия регрессии')
ax1.set_xlabel('Latest industrial data (год)')
ax1.set_ylabel('Latest trade data (год)')
ax1.set_title('Линейная регрессия')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── График 2: реальные vs предсказанные ─────────────────────────────────────
ax2 = axes[1]
ax2.scatter(y_test, y_pred, color='darkorange', alpha=0.7)
# Идеальная линия — если предсказание = реальность
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
ax2.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Идеальное предсказание')
ax2.set_xlabel('Реальные значения')
ax2.set_ylabel('Предсказанные значения')
ax2.set_title('Реальные vs Предсказанные')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Пример предсказания для новой страны

In [ ]:
# Допустим у новой страны последние промышленные данные за 2010 год
# Какой год торговых данных предскажет модель?

new_country = pd.DataFrame({'Latest industrial data': [2005, 2010, 2015, 2020]})
predictions = model.predict(new_country)

print('Примеры предсказаний:')
print('-' * 45)
for ind_year, trade_year in zip(new_country['Latest industrial data'], predictions):
    print(f'  Промышленные данные за {int(ind_year)} → торговые: {trade_year:.1f}')

## 8. Итоги

| Метрика | Значение |
|---|---|
| MAE модели | показывает насколько ошибается в годах |
| R² score | насколько хорошо объясняет данные (0–1) |
| Baseline MAE | ошибка без модели (просто среднее) |

**Как читать R²:**
- `0.0` — модель бесполезна, не лучше среднего
- `0.5` — объясняет половину разброса данных  
- `1.0` — идеальное предсказание (в реальности не бывает)

> **Примечание:** датасет маленький (~100 строк с заполненными значениями), поэтому результаты могут быть скромными. Для серьёзного ML нужно больше данных и больше признаков.